In [2]:
from pathlib import Path
from guppylang import guppy
from guppylang.std.quantum import qubit, x, cx, cz, h, measure, toffoli
from guppylang.std.uncomputation import forget

@guppy(unitary=True)
def use_cx(a: qubit, b: qubit) -> None:
    x(a)
    cx(a, b)

@guppy
def recursive_use_cx(a: qubit, b: qubit, depth: int) -> None:
    if depth > 0:
        use_cx(a, b)
        recursive_use_cx(a, b, depth - 1)

@guppy
def clean_copy(a: qubit, result: qubit, b: bool) -> None:
    temporary = qubit()
    h(a)

    recursive_use_cx(a, temporary, 1)
    cx(temporary, result)
    forget(temporary)



# todo: save the file
output = Path("hugrs/complex_guppy.hugr")
output.parent.mkdir(parents=True, exist_ok=True)
program = clean_copy.compile_function().modules[0]
hugr = program.to_bytes()
with open(output, "wb") as f:
    f.write(hugr)

program.render_dot()

UncomputationError: recursive_use_cx is not controllable

The program with uncomputation here is:

```
a: qubit, result: qubit
temporary = 0
tmp = 0
h(a)
cx(temporary, tmp)
use_cx(a, temporary)
cx(temporary, result)
uncompute(a, temporary) = {cx(a, temporary)}
cx(tmp, temporary)
qfree(temporary)
qfree(tmp)

```